# Function Calling & Structured Outputs

### The two halves of one working agent

An LLM on its own is a very good writer trapped in a sealed room. It has no phone line to your
database, and everything it hands you comes out as prose. Two separate problems, two separate fixes:

| The problem | The fix | Part |
|---|---|---|
| The model **doesn't know** your facts — prices, order status, live weather | **Function calling** — let it ask *your code* for help | **Part 1** |
| The model **answers in prose** — your router/queue/payment code can't consume it | **Structured outputs** — make it fill in a typed contract | **Part 2** |

Most tutorials teach these as unrelated tricks. They aren't. They're the **input side** and the
**output side** of the same agent:

> **Tools are how facts get *in*. Structured outputs are how decisions get *out*.**

In **Part 3** we put them in a single API call and build a support desk that looks up the truth
before it moves real money.

## The map

Everything in this notebook is a piece of this one picture. We build the left arrow in Part 1,
the right arrow in Part 2, and close the loop in Part 3.

```
                THE COPPERLEAF SUPPORT DESK  —  two halves of one agent

  ┌───────────────────────────────────────────────────────────────────────────────┐
  │                                                                               │
  │   "my order 10432 turned up smashed, I paid 49.99, refund me today"           │
  │                          (messy human prose)                                  │
  │                                  │                                            │
  │                                  ▼                                            │
  │        ┌────────────────────────────────────────────────────┐                 │
  │        │                      MODEL                         │                 │
  │        └────────────────────────────────────────────────────┘                 │
  │             │   ▲                                    │                        │
  │    [PART 1] │   │ facts                     [PART 2] │ decision               │
  │       tool  │   │                                    │                        │
  │       calls ▼   │                                    ▼                        │
  │   ┌─────────────────────────┐          TicketDecision(                        │
  │   │ lookup_order            │            order_id       = "10432",            │
  │   │ get_delivery_status     │            category       = "shipping",         │
  │   │ check_refund_policy     │            urgency        = "high",             │
  │   │ get_customer_history    │            action         = "refund",           │
  │   └─────────────────────────┘            refund_amount  = 49.99 )             │
  │        the back office                             │                          │
  │     (your code, your data)                         │                          │
  │                                ┌───────────────────┼───────────────────┐      │
  │                                ▼                   ▼                   ▼      │
  │                          🧭 router          ⏱️ priority queue   💸 refunds     │
  └───────────────────────────────────────────────────────────────────────────────┘

  PART 1  ── FUNCTION CALLING ......... how facts get IN      (tools)
  PART 2  ── STRUCTURED OUTPUTS ....... how decisions get OUT (Pydantic)
  PART 3  ── THE CASE STUDY ........... both, in one call
  PART 4  ── WRAP-UP .................. what each half bought you
```

---

**P0 ▶ SETUP**  ·  P1 function calling  ·  P2 structured outputs  ·  P3 case study  ·  P4 wrap-up

> *key, client, and the model we'll use throughout*

### 0.1 · Install

Run this **once**, then comment it back out. It's commented by default so re-running the
notebook doesn't go to the network every time.

In [1]:
# %pip install -q openai pydantic python-dotenv requests truststore ipython-autotime

### 0.2 · Cell timing

`autotime` prints how long every cell took. In a class that matters — you'll see instantly that a
tool round-trip costs seconds, not milliseconds.

In [2]:
%load_ext autotime

time: 94.5 µs (started: 2026-08-27 08:43:27 +05:30)


### 0.3 · Load the API key

We **never** hardcode API keys. They live in a `.env` file and get loaded at runtime. Treat your
key like a password — if it leaks, anyone can run up charges on your account.

In [3]:
import os, json, textwrap

from dotenv import load_dotenv


# Wrap long model output at 80 columns instead of one endless line.
def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    print(textwrap.fill(text, width=80))


load_dotenv("/Users/shivam13juna/Documents/scaler/GEN_AI_REF/openai_key.env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found! Check the path to your openai_key.env file.")

pretty_print("API key loaded successfully.")

API key loaded successfully.
time: 3.27 ms (started: 2026-08-27 08:43:27 +05:30)


In [4]:
# Optional — only needed if you're behind a VPN or a corporate proxy.
# It makes Python trust the certificates already in your OS keychain.
import truststore

truststore.inject_into_ssl()

time: 30.1 ms (started: 2026-08-27 08:43:27 +05:30)


### 0.4 · The client

The `OpenAI` class is our gateway to every model endpoint. We create it once and reuse it
throughout the notebook.

We're on the **Responses API** (`client.responses.*`) the whole way — it's OpenAI's recommended
API going forward, and it's the one where tools and structured outputs compose cleanly. There's a
translation table from Chat Completions at the end of Part 1 if you're porting old code.

In [5]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
MODEL = "gpt-5-nano"

pretty_print("OpenAI client ready. Model:", MODEL)

OpenAI client ready. Model: gpt-5-nano
time: 334 ms (started: 2026-08-27 08:43:27 +05:30)


---

P0 setup  ·  **P1 ▶ FUNCTION CALLING**  ·  P2 structured outputs  ·  P3 case study  ·  P4 wrap-up

> *how facts get IN  —  the model asks your code for help*

# Part 1 · Function Calling — how facts get IN

LLMs are impressive. They write essays, explain quantum physics, draft emails. But they have three
hard limits:

- 🧊 **Frozen in time** — they don't know anything after their training cutoff
- 🔢 **Bad at precise computation** — they *predict* math answers rather than *computing* them
- 🚫 **Can't take actions** — they can't send email, query a database, or call an API

Let's actually *watch* these fail before we fix them.

## 1.1 · Three experiments, three failures

### 🧪 Experiment 1 — ask about the real world right now

In [6]:
response = client.responses.create(
    model=MODEL,
    input="What is the weather in Bengaluru right now? Give me the exact temperature.",
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"},
)
print(response.output_text)

I don’t have real-time weather data access. Please check a live source like weather.com, a weather app, or a trusted news site for the exact current temperature in Bengaluru. If you want, tell me your preferred source and I can guide you on how to find it.
time: 5.19 s (started: 2026-08-27 08:43:27 +05:30)


☝️ The model either **admits it can't check** — or worse, **invents a plausible-sounding number**.
Either way it's useless to a user who needs the real temperature.

### 🧪 Experiment 2 — ask for precise math

In [7]:
response = client.responses.create(
    model=MODEL,
    input="What is 1247 * 83 + 19 / 3.7? Answer in one line, exact number, 2 decimal places.",
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"},
)
print("Model says   :", response.output_text)
print("Python says  :", 1247 * 83 + 19 / 3.7)

Model says   : 1247 * 83 + 19 / 3.7 = 103,501 + 5.135135... = 103,506.135135... Rounded to 2 decimals: 103506.14
Python says  : 103506.13513513513
time: 1.13 s (started: 2026-08-27 08:43:32 +05:30)


Sometimes it lands, sometimes it's off by thousands. The point isn't the error rate — it's that the
model is **guessing**, not computing. For a banking app, "usually right" is not a feature.

### 🧪 Experiment 3 — ask it to *do* something

In [8]:
response = client.responses.create(
    model=MODEL,
    input="Send an email to john@example.com saying 'Meeting moved to 3pm'.",
    reasoning={"effort": "minimal"},
    text={"verbosity": "low"},
)
print(response.output_text)

I can’t send emails directly. I can help you draft the message and provide steps to send it.

Subject: Meeting moved to 3pm
Body:
Hi John,

Meeting moved to 3pm.

Best regards,
[Your Name]

If you want, tell me your name and I can tailor it or I can provide steps for Gmail/Outlook.
time: 2.21 s (started: 2026-08-27 08:43:33 +05:30)


It can *write* the email. It can't *send* it. It has no access to your mail server, your contacts,
or anything at all outside its own text-generation bubble.

---

### So what do we actually want?

We want the model to be able to say:

> *"I can't compute that myself, but I see a `multiply` function available. Please call
> `multiply(a=1247, b=83)` and tell me the result."*

> *"I can't check live weather, but there's a `get_weather` function. Please call
> `get_weather(location='Bengaluru')` and tell me what it says."*

**That's function calling.** The model doesn't gain superpowers. It learns to **ask for help** from
functions *you* wrote and *you* control.

## 1.2 · The vocabulary

Three terms you'll see everywhere. Nail them down now with one concrete example.

### 🔧 Tool (or Function)
A piece of functionality **you** write and **describe** to the model. The model never sees your
code — only a name, a description, and a schema of the arguments it accepts.

```
Tool name:        get_weather
Description:      "Get the current temperature for a city"
Expected input:   { "location": "Bengaluru" }
```

### 📞 Tool Call (or Function Call)
When the model decides it needs a tool, it doesn't answer with text. It returns a **structured
request**: *"please call this function with these arguments."*

```
Model's response:  function_call → get_weather(location="Bengaluru")
                   (NOT "The weather in Bengaluru is 28°C")
```

### 📦 Tool Call Output (or Function Call Output)
**Your code** actually runs the function, gets a result, and sends it back. The model then uses that
real data to compose its final answer.

```
Your code runs:    get_weather("Bengaluru") → {"temp": 28, "unit": "C"}
You send back:     function_call_output with that result
Model finally:     "The current temperature in Bengaluru is 28°C."
```

**The model proposes. Your app executes. The model summarises.** That's the whole pattern — every
agent framework you'll ever meet is this loop with decoration on top.

## 1.3 · A two-minute Python detour: `**`

You'll see `fn(**args)` in every tool loop from here on, so let's make sure it isn't magic.

In a function *call*, `**` means **"take this dict and expand it into keyword arguments."**

In [9]:
def add_something(a, b):
    print(f"Adding {a} and {b} together...")
    return a + b


# These two lines are IDENTICAL as far as Python is concerned:
add_something(a=7, b=12)
add_something(**{"a": 7, "b": 12})

Adding 7 and 12 together...
Adding 7 and 12 together...


19

time: 1.71 ms (started: 2026-08-27 08:43:36 +05:30)


That's not a dict being passed as one argument — it's **unpacked** into named parameters, because
the dict keys (`"a"`, `"b"`) match the parameter names (`a`, `b`).

**Why this matters here:** the model hands us arguments as a JSON string like `{"a": 7, "b": 12}`.
We `json.loads` it into a dict, and then `add(**args)` calls our real Python function with them.
That single line is the bridge between the model's request and your code.

> Related: `*` unpacks a *sequence* into positional arguments — `add_something(*[7, 12])`.

## 1.4 · Anatomy of a tool definition

Here's the contract you hand the model. Every field earns its place:

```python
{
    "type": "function",          # always "function" for your own tools
    "name": "add",               # what the model calls it by
    "description": "Add two numbers together.",   # helps the model decide WHEN to use it
    "parameters": {              # JSON Schema describing the arguments
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],           # which fields are mandatory
        "additionalProperties": False,    # no extra fields (required by strict mode)
    },
    "strict": True,              # enforce the schema — reject anything that doesn't match
}
```

- The **`description`** is the part the model reads to decide *whether this tool is relevant*.
  A vague description is the #1 cause of "why didn't it call my tool?"
- The **`parameters`** schema tells it *exactly what arguments to generate*.
- **`strict: True`** makes the API guarantee the arguments match your schema.

> The model never sees your Python function. It only ever sees this JSON.

Let's give it an `add` tool and watch what comes back.

In [10]:
add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="What is 7 + 12?",
    tools=[add_tool],
)

print("Output items from the model:")
print("-" * 45)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"    name      : {item.name}")
        print(f"    arguments : {item.arguments}")
        print(f"    call_id   : {item.call_id}")

print(f"\noutput_text: {response.output_text!r}   <- empty! it didn't answer, it ASKED")

Output items from the model:
---------------------------------------------
  Type: reasoning
  Type: function_call
    name      : add
    arguments : {"a":7,"b":12}
    call_id   : call_pk145L3rbMjPjT1Fhw8YLpPd

output_text: ''   <- empty! it didn't answer, it ASKED
time: 1.79 s (started: 2026-08-27 08:43:36 +05:30)


🎯 **Look at that.** The model did *not* reply `"19"`. It returned:

- `type: "function_call"` — a request, not an answer
- `name: "add"` — it picked our function
- `arguments: {"a": 7, "b": 12}` — it worked out the arguments from plain English
- `call_id: "call_..."` — the ticket number we'll quote when we hand back the result

And `output_text` is **empty**. The model is waiting for us.

### Does having a tool force it to use one?

No. Same tool, a question that has nothing to do with arithmetic:

In [11]:
response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="In one sentence, why is the Earth round?",
    tools=[add_tool],
    reasoning={"effort": "minimal"},
)

print("Item types:", [item.type for item in response.output])
pretty_print("Response:", response.output_text)

Item types: ['reasoning', 'message']
Response: Earth is round because gravity pulls matter into a shape that
minimizes potential energy, forming a sphere.
time: 1.01 s (started: 2026-08-27 08:43:37 +05:30)


`message`, not `function_call`. **The model decides.** You're offering capabilities, not writing an
if-statement — which is exactly why the `description` field matters so much.

## 1.5 · The complete flow, end to end

The model asked. Now we answer. Five steps — and every agent you ever build is these five steps in
a loop.

```
  1.  YOU  ──▶  API        user question  +  tool definitions
  2.            API ──▶    model returns function_call items (not text!)
  3.  YOU execute the function(s) yourself, with the model's arguments
  4.  YOU  ──▶  API        send back a function_call_output for each call
  5.            API ──▶    model writes the final answer using the real result
```

Let's do all five, one cell at a time.

In [12]:
# ── Step 0: our actual Python function. The model never sees this. ──
def add(a, b):
    return a + b


# ── Step 1: call the API, offering the tool ──
response = client.responses.create(
    model=MODEL,
    instructions="Always use the add tool for math. Never compute yourself.",
    input="What is 7 + 12?",
    tools=[add_tool],
)

# ── Step 2: the model returns a function_call, not text ──
print("Step 2 — model returned:")
for item in response.output:
    if item.type == "function_call":
        print(f"   {item.name}({item.arguments})   call_id={item.call_id}")

Step 2 — model returned:
   add({"a":7,"b":12})   call_id=call_1ofesVnTqVZd3YkQabCcsJLH
time: 2.32 s (started: 2026-08-27 08:43:38 +05:30)


In [13]:
# ── Step 3: WE execute it, with the model's arguments ──
function_call = [item for item in response.output if item.type == "function_call"][0]
args = json.loads(function_call.arguments)   # '{"a":7,"b":12}'  ->  {'a': 7, 'b': 12}

result = add(**args)                          # the ** from section 1.3, doing real work
print(f"Step 3 — we ran add(**{args}) -> {result}")

Step 3 — we ran add(**{'a': 7, 'b': 12}) -> 19
time: 317 µs (started: 2026-08-27 08:43:41 +05:30)


In [14]:
# ── Step 4: hand the result back, quoting the call_id so it knows which call this answers ──
tool_outputs = [
    {
        "type": "function_call_output",
        "call_id": function_call.call_id,
        "output": str(result),          # must be a STRING
    }
]
print("Step 4 — sending back:", tool_outputs)

# ── Step 5: the model writes the final answer using our real number ──
# previous_response_id links this call to the earlier one, so we don't have to
# re-send the question or the model's own reasoning.
final = client.responses.create(
    model=MODEL,
    instructions="Always use the add tool for math. Never compute yourself.",
    previous_response_id=response.id,
    input=tool_outputs,
    tools=[add_tool],
)

print("Step 5 — final answer:", final.output_text)

Step 4 — sending back: [{'type': 'function_call_output', 'call_id': 'call_1ofesVnTqVZd3YkQabCcsJLH', 'output': '19'}]


Step 5 — final answer: The result is 19.
time: 944 ms (started: 2026-08-27 08:43:41 +05:30)


That's the entire mechanism. Note what `previous_response_id` bought us: we sent back **only** the
tool result — not the original question, not the conversation. The API already has it.

## 1.6 · More than one tool, and more than one round

Real questions need several tools, and sometimes the *arguments* for the second tool depend on the
*results* of the first. Let's give the model four arithmetic tools and one expression that can't be
done in a single pass.

In [15]:
# Four tools. Same shape as add_tool, so read one and skim the rest.
sub_tool = {
    "type": "function",
    "name": "subtract",
    "description": "Subtract b from a and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

div_tool = {
    "type": "function",
    "name": "divide",
    "description": "Divide a by b and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


def subtract(a, b):
    return a - b


def multiply(a, b):
    return a * b


def divide(a, b):
    if b == 0:
        return "Error: Division by zero"
    return a / b


# Two lookup tables we'll pass around together: what the model sees, and what we run.
TOOLS = [add_tool, sub_tool, mul_tool, div_tool]
DISPATCH = {"add": add, "subtract": subtract, "multiply": multiply, "divide": divide}

print("Offering", len(TOOLS), "tools:", list(DISPATCH))

Offering 4 tools: ['add', 'subtract', 'multiply', 'divide']
time: 865 µs (started: 2026-08-27 08:43:42 +05:30)


### Round 1 — what can it do straight away?

`1247 * 83 + 19 / 3.7` needs three operations, but the multiply and the divide are **independent**,
so it can ask for both at once. The final `add` has to wait for their results.

In [16]:
DEV_POLICY = "Use the tools for any math. Never compute math yourself."

response = client.responses.create(
    model=MODEL,
    instructions=DEV_POLICY,
    input="What is 1247 * 83 + 19 / 3.7?",
    tools=TOOLS,
    reasoning={"effort": "minimal"},
)

print("Round 1 — the model asked for:")
for item in response.output:
    if item.type == "function_call":
        print(f"   {item.name}({item.arguments})")

Round 1 — the model asked for:
   multiply({"a":1247,"b":83})
   divide({"a":19,"b":3.7})
time: 1.24 s (started: 2026-08-27 08:43:42 +05:30)


In [17]:
# We run BOTH calls and send BOTH results back in one go.
tool_outputs = []
for call in response.output:
    if call.type != "function_call":
        continue
    args = json.loads(call.arguments)
    result = DISPATCH[call.name](**args)      # look the function up by name, then unpack the args
    print(f"   we ran {call.name}(**{args}) -> {result}")
    tool_outputs.append(
        {"type": "function_call_output", "call_id": call.call_id, "output": str(result)}
    )

response = client.responses.create(
    model=MODEL,
    instructions=DEV_POLICY,
    previous_response_id=response.id,
    input=tool_outputs,
    tools=TOOLS,
    reasoning={"effort": "minimal"},
)

print("\nRound 2 — the model asked for:")
for item in response.output:
    if item.type == "function_call":
        print(f"   {item.name}({item.arguments})")

   we ran multiply(**{'a': 1247, 'b': 83}) -> 103501
   we ran divide(**{'a': 19, 'b': 3.7}) -> 5.135135135135135



Round 2 — the model asked for:
   add({"a":103501,"b":5.135135135135135})
time: 1.27 s (started: 2026-08-27 08:43:43 +05:30)


👀 **There it is.** Round 2's `add` arguments are *round 1's answers*. The model chained the calls
by itself — we never told it the order of operations.

This is why a single request/response isn't enough: you need a **loop** that keeps going until the
model stops asking for tools.

In [18]:
# One more round to finish it off.
tool_outputs = []
for call in response.output:
    if call.type != "function_call":
        continue
    args = json.loads(call.arguments)
    result = DISPATCH[call.name](**args)
    print(f"   we ran {call.name}(**{args}) -> {result}")
    tool_outputs.append(
        {"type": "function_call_output", "call_id": call.call_id, "output": str(result)}
    )

final = client.responses.create(
    model=MODEL,
    instructions=DEV_POLICY,
    previous_response_id=response.id,
    input=tool_outputs,
    tools=TOOLS,
    reasoning={"effort": "minimal"},
)

print("\nRound 3 — no more tool calls. Final answer:")
print("  ", final.output_text)
print("  Python agrees:", 1247 * 83 + 19 / 3.7)

   we ran add(**{'a': 103501, 'b': 5.135135135135135}) -> 103506.13513513513



Round 3 — no more tool calls. Final answer:
   103506.13513513513
  Python agrees: 103506.13513513513
time: 840 ms (started: 2026-08-27 08:43:44 +05:30)


## 1.7 · The loop, written once

You've now done every round by hand, so nothing below is new — it's the same three cells with a
`while` around them. **This is the only helper function in the notebook**, and we'll reuse it
unchanged in Part 3, so it's worth reading line by line.

The one thing to notice: we call `client.responses.**parse**` rather than `.create`. Without a
`text_format` argument it behaves exactly like `.create` — but it's the same function we'll need in
Part 3 when we *do* want a typed object back. One loop, both jobs.

In [19]:
def run_tool_loop(user_message, tools, dispatch, instructions, text_format=None):
    # Ask -> run whatever tools it asks for -> hand the results back -> repeat.
    # Returns the final response object:
    #   text_format given  ->  read response.output_parsed  (a validated Pydantic object)
    #   text_format None   ->  read response.output_text    (prose)
    schema = {"text_format": text_format} if text_format else {}

    response = client.responses.parse(
        model=MODEL,
        instructions=instructions,
        input=[{"role": "user", "content": user_message}],
        tools=tools,
        **schema,
    )

    for round_no in range(1, 7):                 # hard cap, so a confused model can't spin forever
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:                            # it stopped asking -> this is the answer
            return response

        tool_outputs = []
        for call in calls:
            args = json.loads(call.arguments)
            result = dispatch[call.name](**args)
            print(f"   round {round_no}: {call.name}({args}) -> {result}")
            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(result),   # json.dumps handles dicts, numbers and strings
                }
            )

        response = client.responses.parse(
            model=MODEL,
            instructions=instructions,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
            **schema,
        )

    still_wants = [item.name for item in response.output if item.type == "function_call"]
    raise RuntimeError(f"Still asking for tools after 6 rounds: {still_wants}. "
                       "Usually means a tool result didn't answer the question it was asked.")

time: 814 µs (started: 2026-08-27 08:43:45 +05:30)


In [20]:
# The same question, now in one call.
final = run_tool_loop(
    "What is 1247 * 83 + 19 / 3.7?",
    tools=TOOLS,
    dispatch=DISPATCH,
    instructions=DEV_POLICY,
)
print("\nFinal:", final.output_text)

   round 1: multiply({'a': 1247, 'b': 83}) -> 103501
   round 1: divide({'a': 19, 'b': 3.7}) -> 5.135135135135135


   round 2: add({'a': 103501, 'b': 5.135135135135135}) -> 103506.13513513513



Final: 103506.13513513513
time: 5.79 s (started: 2026-08-27 08:43:45 +05:30)


## 1.8 · Swapping a fake tool for a real one

Everything so far used toy functions. Let's do the weather properly, with
[Open-Meteo](https://open-meteo.com/) — a free API that needs **no key**.

Two HTTP calls are involved:
1. **Geocoding** — turn `"Bengaluru"` into latitude/longitude.
2. **Forecast** — fetch the current conditions at those coordinates.

Watch what *doesn't* change: **the tool definition**. The model's side of the contract is identical
whether the function returns a hardcoded dict or hits a live API. That separation is the whole point.

One thing that *does* matter is what your function **returns**. Notice both versions below name the
city in the result. Hand back a bare `"18°C"` for two different cities and the model can't tell which
is which — so it asks again, and again, until your loop's round cap fires. Tool results have to be
self-describing.

In [21]:
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City name, e.g. 'Bengaluru', 'London'"},
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}


# Version 1 — a fake. Good enough to develop the loop against.
def get_weather(location):
    # Note the city name in every value. A tool result must say what it is ABOUT --
    # hand back two bare strings like "18°C" and the model can't tell which city is which,
    # so it just asks again. Self-describing outputs are not decoration.
    fake_data = {
        "Bengaluru": "Bengaluru: 28°C, sunny",
        "Paris": "Paris: 18°C, cloudy",
        "London": "London: 14°C, rain",
    }
    return fake_data.get(location, f"No data for {location}")


final = run_tool_loop(
    "What's the weather like in Paris and London?",
    tools=[weather_tool],
    dispatch={"get_weather": get_weather},
    instructions="Use get_weather for any weather question. Never guess.",
)
print("\n", final.output_text)

   round 1: get_weather({'location': 'Paris'}) -> Paris: 18°C, cloudy
   round 1: get_weather({'location': 'London'}) -> London: 14°C, rain



 Here’s the current weather:

- Paris: 18°C, cloudy
- London: 14°C, rain

Want a short-term forecast or alerts for either city?
time: 5.91 s (started: 2026-08-27 08:43:51 +05:30)


In [22]:
import requests

# A VPN-friendly session: ignore HTTP(S)_PROXY / NO_PROXY environment variables.
SESSION = requests.Session()
SESSION.trust_env = False

GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

# WMO weather codes -> human labels
WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "light rain", 63: "moderate rain", 65: "heavy rain",
    71: "light snow", 73: "moderate snow", 75: "heavy snow",
    80: "rain showers", 81: "heavy rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm w/ hail", 99: "severe thunderstorm w/ hail",
}


# Version 2 — the real thing. Same name, same signature, same tool definition.
def get_weather(location):
    # Step 1: city name -> coordinates
    r = SESSION.get(
        GEOCODE_URL,
        params={"name": location, "count": 1, "language": "en", "format": "json"},
        timeout=10,
    )
    r.raise_for_status()
    results = r.json().get("results") or []
    if not results:
        return f"No data for {location}"
    place = results[0]
    lat, lon = place["latitude"], place["longitude"]
    resolved = ", ".join(filter(None, [place.get("name"), place.get("admin1"), place.get("country")]))

    # Step 2: coordinates -> current conditions
    r = SESSION.get(
        FORECAST_URL,
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
            "timezone": "auto",
        },
        timeout=10,
    )
    r.raise_for_status()
    cur = r.json().get("current", {})
    condition = WEATHER_CODES.get(cur.get("weather_code"), "unknown")
    return (
        f"{resolved}: {cur.get('temperature_2m')}°C, {condition}, "
        f"humidity {cur.get('relative_humidity_2m')}%, wind {cur.get('wind_speed_10m')} km/h"
    )


print(get_weather("Bengaluru"))

Bengaluru, Karnataka, India: 23.0°C, overcast, humidity 76%, wind 18.5 km/h
time: 1.56 s (started: 2026-08-27 08:43:57 +05:30)


In [23]:
# Identical call to the fake version above — only the function body changed.
final = run_tool_loop(
    "What's the weather like in Bengaluru and London right now?",
    tools=[weather_tool],
    dispatch={"get_weather": get_weather},
    instructions="Use get_weather for any weather question. Never guess.",
)
pretty_print(final.output_text)

   round 1: get_weather({'location': 'Bengaluru'}) -> Bengaluru, Karnataka, India: 23.0°C, overcast, humidity 76%, wind 18.5 km/h


   round 1: get_weather({'location': 'London'}) -> London, England, United Kingdom: 20.6°C, light drizzle, humidity 85%, wind 11.2 km/h


Here’s the current weather:  - Bengaluru, Karnataka, India: 23.0°C, overcast,
humidity 76%, wind 18.5 km/h - London, England, United Kingdom: 20.6°C, light
drizzle, humidity 85%, wind 11.2 km/h
time: 11.3 s (started: 2026-08-27 08:43:58 +05:30)


Compare that to Experiment 1 at the top of Part 1, where the model either refused or made something
up. Same model, same question — one tool's difference.

## 1.9 · Recap

| Without function calling | With function calling |
|---|---|
| *"The weather is probably around 25°C"* | calls `get_weather("Bengaluru")` → the **real** number |
| *"1247 × 83 = 103,601"* (wrong) | calls `multiply`, `divide`, `add` → **exact** |
| *"I can't send emails"* | calls `send_email(...)` → the email **actually goes** |

**The model's job changed.** Instead of pretending to know everything, it now decides *which tool to
call, with what arguments*. Your code does the rest — which also means your code keeps the
permissions, the rate limits, and the audit log.

### Responses API vs Chat Completions

If you're porting older code, here's the translation table:

| Chat Completions (old) | Responses API (used here) |
|---|---|
| `client.chat.completions.create()` | `client.responses.create()` / `.parse()` |
| `messages=[{role, content}]` | `input=[{role, content}]`, or just a string |
| system message inside `messages` | `instructions="..."` parameter |
| `response.choices[0].message.content` | `response.output_text` |
| `response.choices[0].message.tool_calls` | `response.output` items with `type == "function_call"` |
| `{role:"tool", tool_call_id:…, content:…}` | `{type:"function_call_output", call_id:…, output:…}` |
| resend the whole message list each turn | `previous_response_id=…` |
| `response_format={"type":"json_schema",…}` | `text_format=YourPydanticModel` (Part 2) |

## 1.10 · What function calling still does **not** give you

Our weather agent now knows real facts. But look closely at what it hands back:

In [24]:
print(repr(final.output_text[:300]))

'Here’s the current weather:\n\n- Bengaluru, Karnataka, India: 23.0°C, overcast, humidity 76%, wind 18.5 km/h\n- London, England, United Kingdom: 20.6°C, light drizzle, humidity 85%, wind 11.2 km/h'
time: 402 µs (started: 2026-08-27 08:44:10 +05:30)


**Prose.** A wall of text, formatted however the model felt like today.

That's fine for a human reading a chat window. But suppose the next thing downstream isn't a human —
suppose it's a dashboard that needs `{"city": "Bengaluru", "temp_c": 28.4}`. You're now back to
regex-ing sentences and hoping the wording doesn't drift.

Function calling fixed the **input** side: the model can reach your world. It did nothing at all for
the **output** side: your world still can't reliably read the model.

That's Part 2.

---

P0 setup  ·  P1 function calling  ·  **P2 ▶ STRUCTURED OUTPUTS**  ·  P3 case study  ·  P4 wrap-up

> *how decisions get OUT  —  your code stops trusting prose*

# Part 2 · Structured Outputs — how decisions get OUT

In Part 1 we taught the model to **ask for help**. Now the other half of the problem:

> When the model *answers*, **how do we trust the shape of what comes back?**

An LLM speaks *human*. Your code speaks *data* — typed fields, numbers, enums, things you can `if`,
`for` and `+` on. The gap between those two is where production systems quietly break.

## Meet Copperleaf

**Copperleaf** is an online homeware store — mugs, kettles, cast-iron pans. Their support inbox
gets free text like this:

> *"hey my order 10432 arrived smashed, the mug is in pieces. I paid 49.99 for it, want my money
> back asap"*

Their automation has to turn that into **structured data** and feed three downstream systems:

| Field | Used by | What happens if it's wrong |
|---|---|---|
| `category` | 🧭 the **router** (billing / shipping / technical / …) | ticket lands in the wrong team's queue — or nowhere |
| `urgency` | ⏱️ the **priority queue** | an angry customer waits days; a trivial issue jumps the line |
| `refund_amount` | 💸 the **refund processor** (moves *real money*) | we refund the wrong amount — **literally lose money** |

Three systems. Three different ways to bleed money. Let's watch it happen, then fix it for good.

In [25]:
# One messy, realistic customer email we'll reuse for the whole of Part 2.
# Note the signature — we'll need the name and address later.
customer_email = (
    "hey, my order 10432 turned up completely smashed — the ceramic mug is in pieces. "
    "I paid 49.99 for it and honestly I'm pretty annoyed, this is the second time. "
    "I just want my money back asap, can you sort this out today?\n\n"
    "Thanks, Sam Rivera (sam.rivera@example.com)"
)
print(customer_email)

hey, my order 10432 turned up completely smashed — the ceramic mug is in pieces. I paid 49.99 for it and honestly I'm pretty annoyed, this is the second time. I just want my money back asap, can you sort this out today?

Thanks, Sam Rivera (sam.rivera@example.com)
time: 311 µs (started: 2026-08-27 08:44:10 +05:30)


## 2.1 · The problem — the model answers in prose, not in fields

Let's extract the ticket the way a beginner would: just ask nicely.

In [26]:
response = client.responses.create(
    model=MODEL,
    input=f"Extract the category, urgency and refund amount from this customer message:\n\n{customer_email}",
    reasoning={"effort": "minimal"},
)
print(response.output_text)

- Category: Product damaged / defective (damaged item)
- Urgency: Urgent (wants refund today)
- Refund amount: 49.99
time: 1.21 s (started: 2026-08-27 08:44:10 +05:30)


That reads beautifully — **to a human**. Now try to *use* it in code.

- Where's the number? Is it `49.99`, `$49.99`, or "around fifty dollars"?
- Is the category `shipping`, or `"damaged item / shipping"`?
- Re-run the cell a few times and the wording **drifts** every time.

Your code can't `if category == ...` against a moving target. Let's make the pain concrete — grab
the refund amount and try to do arithmetic on it.

In [27]:
# Copperleaf's refund processor adds a 5% handling fee — i.e. it has to do MATH on the amount.
response = client.responses.create(
    model=MODEL,
    input=f"In one short line, what refund does this customer want?\n\n{customer_email}",
    reasoning={"effort": "minimal"},
)
raw_amount = response.output_text
print("Model said:", repr(raw_amount))

try:
    total_with_fee = raw_amount * 1.05          # string * float
    print("Refund + fee:", total_with_fee)
except Exception as e:
    print("💥 Downstream code blew up:", type(e).__name__, "-", e)

Model said: 'They want a full refund of $49.99 for a shattered mug and for it to be resolved today.'
💥 Downstream code blew up: TypeError - can't multiply sequence by non-int of type 'float'
time: 1.29 s (started: 2026-08-27 08:44:11 +05:30)


## 2.2 · The "just ask for JSON" trap

*"Easy,"* you say, *"I'll just tell it to return JSON."* Let's try it — ten times in parallel, so we
can see the variance rather than getting lucky once.

In [28]:
import concurrent.futures


def fetch_ticket(i):
    prompt = (
        "Extract the support ticket as JSON with keys: category, urgency, refund_amount.\n"
        f"Customer message:\n{customer_email}"
    )
    response = client.responses.create(model=MODEL, input=prompt, reasoning={"effort": "minimal"})
    return i, response.output_text


with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
    futures = [pool.submit(fetch_ticket, i) for i in range(10)]
    for future in concurrent.futures.as_completed(futures):
        i, output = future.result()
        print(f"[{i}]", output.replace("\n", " "))

[0] {   "category": "refund",   "urgency": "high",   "refund_amount": 49.99 }
[2] {   "category": "product_damage",   "urgency": "high",   "refund_amount": 49.99 }
[3] {   "category": "damaged_item",   "urgency": "high",   "refund_amount": 49.99 }


[4] {   "category": "refund_request",   "urgency": "high",   "refund_amount": 49.99 }


[1] {   "category": "Shipping/Delivery Issue",   "urgency": "High",   "refund_amount": 49.99 }


[5] {   "category": "damaged_item",   "urgency": "high",   "refund_amount": 49.99 }
[6] {   "category": "Refund",   "urgency": "High",   "refund_amount": 49.99 }


[8] {   "category": "damaged_item",   "urgency": "high",   "refund_amount": 49.99 }


[7] {   "category": "damaged_item",   "urgency": "high",   "refund_amount": 49.99 }
[9] {   "category": "refund_request",   "urgency": "high",   "refund_amount": 49.99 }
time: 3.42 s (started: 2026-08-27 08:44:12 +05:30)


Sometimes that's clean JSON. Often it isn't:

- 🪧 wrapped in <code>```json … ```</code> fences, so `json.loads` chokes
- 💬 a friendly sentence in front of it (*"Sure! Here's the data:"*)
- 🔢 `refund_amount` comes back as `"49.99"` (a **string**) or `"$49.99"`
- 🧭 `category` is `"damaged_item"` — a label your router has **never heard of**
- 🕳️ a key silently **missing**

Look at the run above and count how many distinct `category` values you got for **one identical
email**. Each variant is a production incident waiting to happen. Let's parse it the fragile way and
watch.

In [29]:
VALID_CATEGORIES = {"billing", "shipping", "technical", "account", "other"}


def naive_pipeline(model_text):
    data = json.loads(model_text)          # 💥 #1: dies on ``` fences or a prose preamble
    category = data["category"]            # 💥 #2: KeyError if the key is missing
    amount = data["refund_amount"]

    if category not in VALID_CATEGORIES:   # 💥 #3: an invented category -> silent misroute
        print(f"   ⚠️  Unknown category {category!r} -> ticket dropped on the floor")
    else:
        print(f"   🧭 routed to: {category}")

    fee = amount * 1.05                    # 💥 #4: "49.99" * 1.05 explodes; "$49.99" is worse
    print(f"   💸 issuing refund (with fee): {fee}")


try:
    naive_pipeline(output)                 # the last output from the parallel run above
except Exception as e:
    print("💥 pipeline crashed:", type(e).__name__, "-", e)

   ⚠️  Unknown category 'refund_request' -> ticket dropped on the floor
   💸 issuing refund (with fee): 52.48950000000001
time: 546 µs (started: 2026-08-27 08:44:16 +05:30)


A crash is the **lucky** outcome — at least you find out.

The genuinely expensive bug is the one that **looks fine and runs**. Imagine the model returns
`4999` — it read "49.99" as cents, or just slipped a decimal. It's a valid number. `json.loads` is
happy. Your refund processor is happy.

In [30]:
looks_fine = '{"category": "shipping", "urgency": "high", "refund_amount": 4999}'

data = json.loads(looks_fine)              # ✅ no error
refund = data["refund_amount"] * 1.05      # ✅ no error
print(f"💸 Wired ${refund:,.2f} to the customer")     # 😱 should have been ~$52.49
print("Nothing crashed. That's exactly the problem.")

💸 Wired $5,248.95 to the customer
Nothing crashed. That's exactly the problem.
time: 234 µs (started: 2026-08-27 08:44:16 +05:30)


## 2.3 · Enter Pydantic — turn "what we asked for" into a contract

Step back and look at what we kept doing in 2.1 and 2.2: we *described*, in English, the shape we
wanted — "give me `category`, `urgency`, `refund_amount`." The model treated that as a
**suggestion**, and our `json.loads` blindly trusted whatever came back.

**Pydantic** lets us write that shape down **once**, as a real Python class — not a sentence in a
prompt, but an enforceable **contract**. The plan:

1. **Now:** define a `SupportTicket` class = *"this is what a valid ticket looks like."*
2. **Now:** run the model's output through it — junk gets rejected *before* it reaches the refund code.
3. **In 2.7:** hand that *same class* to the API, so the model is **forced** to fill it in.

So `SupportTicket` isn't a new toy. It's the codified version of the keys we've been begging the
model for, and it's the object that carries the rest of this notebook.

In [31]:
from pydantic import BaseModel


# The contract: a valid Copperleaf ticket is EXACTLY these three typed fields.
class SupportTicket(BaseModel):
    category: str
    urgency: str
    refund_amount: float


# Picture the model returning clean JSON and us doing json.loads(...) -> this dict.
# Instead of TRUSTING it the way naive_pipeline did, we VALIDATE it against the contract:
model_output = {"category": "shipping", "urgency": "high", "refund_amount": 49.99}

ticket = SupportTicket.model_validate(model_output)     # the gate the model's output must pass
print("✅ validated:", ticket)

# refund_amount is now a guaranteed float, so the math that crashed in 2.1 just works:
print("   refund + 5% fee:", round(ticket.refund_amount * 1.05, 2))

✅ validated: category='shipping' urgency='high' refund_amount=49.99
   refund + 5% fee: 52.49
time: 712 µs (started: 2026-08-27 08:44:16 +05:30)


In [32]:
from pydantic import ValidationError

# Reality #1 from 2.2: the model returned the amount as a STRING. Pydantic coerces it.
print(SupportTicket.model_validate({"category": "billing", "urgency": "low", "refund_amount": "49.99"}))

# Reality #2: the model returned unparseable prose. Pydantic rejects it — loudly,
# before a single rupee can move.
try:
    SupportTicket.model_validate(
        {"category": "billing", "urgency": "low", "refund_amount": "around fifty bucks"}
    )
except ValidationError as e:
    print("\n🚫 Pydantic blocked the model's junk:\n", e)

category='billing' urgency='low' refund_amount=49.99

🚫 Pydantic blocked the model's junk:
 1 validation error for SupportTicket
refund_amount
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='around fifty bucks', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/float_parsing
time: 336 µs (started: 2026-08-27 08:44:16 +05:30)


## 2.4 · Tightening the contract — each rule kills a real bug from 2.2

A plain `float` still accepts the model's *worst* outputs: the `4999` that wired **$5,249**, and the
invented category `"damaged_item"`. Let's make those **impossible to represent**.

We're not touring Pydantic's API here. Every rule below exists because we already watched it fail.

### `Literal` + `Field`

Copperleaf's router knows exactly five categories, and refunds have a sane ceiling — nothing in the
catalogue costs more than $500. Encode both into the **type** itself.

In [33]:
from typing import Literal

from pydantic import Field


class SupportTicket(BaseModel):
    # Only these exact strings exist -> the model cannot invent a category
    category: Literal["billing", "shipping", "technical", "account", "other"]
    urgency: Literal["low", "medium", "high"]
    # Money is bounded -> the 100x decimal slip can't get through
    refund_amount: float = Field(ge=0, le=500)


# The invented category from 2.2:
try:
    SupportTicket.model_validate({"category": "damaged_item", "urgency": "high", "refund_amount": 49.99})
except ValidationError as e:
    print("🚫 bad category :", e.errors()[0]["msg"])

# The EXACT payload from the silent-bug cell that wired $5,249:
try:
    SupportTicket.model_validate({"category": "shipping", "urgency": "high", "refund_amount": 4999})
except ValidationError as e:
    print("🚫 absurd amount:", e.errors()[0]["msg"])

# A legitimate ticket still passes untouched:
print("✅", SupportTicket.model_validate(
    {"category": "shipping", "urgency": "high", "refund_amount": 49.99}))

🚫 bad category : Input should be 'billing', 'shipping', 'technical', 'account' or 'other'
🚫 absurd amount: Input should be less than or equal to 500
✅ category='shipping' urgency='high' refund_amount=49.99
time: 1.06 ms (started: 2026-08-27 08:44:16 +05:30)


### 2.5 · Custom rules and nesting — when types aren't enough

Two more things real data does:

- A Copperleaf **order id** is always 5 digits. No built-in type says that — a **custom validator** does.
- A ticket carries a nested **customer** (name + email). Pydantic models **compose**, so the model's
  output stays honest all the way down.

In [34]:
from pydantic import field_validator


class Customer(BaseModel):
    name: str
    email: str


class SupportTicket(BaseModel):
    category: Literal["billing", "shipping", "technical", "account", "other"]
    urgency: Literal["low", "medium", "high"]
    refund_amount: float = Field(ge=0, le=500)
    order_id: str = Field(description="The 5-digit Copperleaf order number")
    customer: Customer                                   # <- a nested model

    @field_validator("order_id")
    @classmethod
    def must_be_5_digits(cls, v):
        if not (v.isdigit() and len(v) == 5):
            raise ValueError(f"order_id must be exactly 5 digits, got {v!r}")
        return v


# A malformed order id is caught:
try:
    SupportTicket.model_validate({
        "category": "shipping", "urgency": "high", "refund_amount": 49.99,
        "order_id": "ABC", "customer": {"name": "Sam", "email": "sam@x.com"},
    })
except ValidationError as e:
    print("🚫", e.errors()[0]["msg"])

# A well-formed payload validates, and the nested dict becomes a real Customer object:
ticket = SupportTicket.model_validate({
    "category": "shipping", "urgency": "high", "refund_amount": 49.99,
    "order_id": "10432", "customer": {"name": "Sam", "email": "sam@x.com"},
})
print("✅", ticket)
print("   customer email:", ticket.customer.email)       # nested access, fully typed

🚫 Value error, order_id must be exactly 5 digits, got 'ABC'
✅ category='shipping' urgency='high' refund_amount=49.99 order_id='10432' customer=Customer(name='Sam', email='sam@x.com')
   customer email: sam@x.com
time: 2.46 ms (started: 2026-08-27 08:44:16 +05:30)


## 2.6 · Closing the loop — validate what the model *actually* sends

Everything above used dicts **we** wrote by hand to mimic the model. Let's wire it to the real
thing: call the model the fragile way from 2.2, then pour its reply straight into our contract with
`model_validate_json` (which parses the JSON **and** validates, in one step).

This is the honest "manual" way to combine an LLM with Pydantic.

In [35]:
raw = client.responses.create(
    model=MODEL,
    input=(
        "Reply with ONLY JSON for these keys: category, urgency, refund_amount, order_id, "
        "and customer (an object with name and email).\n\n"
        f"Customer message:\n{customer_email}"
    ),
    reasoning={"effort": "minimal"},
).output_text

print("Raw model text:\n", raw, "\n")

try:
    ticket = SupportTicket.model_validate_json(raw)      # json.loads + validate, in one call
    print("✅ validated ticket:", ticket)
except Exception as e:
    print("💥 Couldn't validate the raw text:", type(e).__name__, "-", str(e)[:300])

Raw model text:
 {
  "category": "refund_request",
  "urgency": "high",
  "refund_amount": 49.99,
  "order_id": "10432",
  "customer": {
    "name": "Sam Rivera",
    "email": "sam.rivera@example.com"
  }
  } 

💥 Couldn't validate the raw text: ValidationError - 1 validation error for SupportTicket
category
  Input should be 'billing', 'shipping', 'technical', 'account' or 'other' [type=literal_error, input_value='refund_request', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error
time: 1.55 s (started: 2026-08-27 08:44:16 +05:30)


Whether that worked depends entirely on the mood the model was in. The fragility is still all here:

- if it adds a sentence or <code>```json</code> fences, `model_validate_json` never even runs;
- it can still omit a field or invent a category, and we only find out **after** paying for the
  round-trip, by catching an exception.

We're validating **after the fact**. Wouldn't it be better to force the model to emit our schema in
the first place?

## 2.7 · `responses.parse` — hand the model your class

Same `SupportTicket` class we just built, now given to the API via **`text_format=`**:

- The SDK turns our class into a **strict JSON schema** and constrains the model's output to match
  it — no fences, no prose, no missing keys, no invented categories.
- We get back a **ready-made `SupportTicket` instance** at `response.output_parsed`. The
  `json.loads` + `model_validate` dance from 2.6 happens for us, and it can't fail the way it did.

It's the manual bridge from 2.6, made bulletproof. Same messy email — watch.

In [36]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content":
            "Extract the support ticket from the customer message. Infer urgency from the tone. "
            "refund_amount is what the customer paid / wants back (0 if they aren't asking for one)."},
        {"role": "user", "content": customer_email},
    ],
    text_format=SupportTicket,          # <- hand the model OUR class
)

ticket = response.output_parsed          # <- already a validated SupportTicket, not text
print("Type of result:", type(ticket).__name__)
print(ticket, "\n")
print("category :", ticket.category, " (a guaranteed-valid Literal)")
print("urgency  :", ticket.urgency)
print("refund   :", ticket.refund_amount, "->", type(ticket.refund_amount).__name__)
print("order id :", ticket.order_id)
print("customer :", ticket.customer.name, "<" + ticket.customer.email + ">")

Type of result: SupportTicket
category='shipping' urgency='high' refund_amount=49.99 order_id='10432' customer=Customer(name='Sam Rivera', email='sam.rivera@example.com') 

category : shipping  (a guaranteed-valid Literal)
urgency  : high
refund   : 49.99 -> float
order id : 10432
customer : Sam Rivera <sam.rivera@example.com>
time: 8.72 s (started: 2026-08-27 08:44:17 +05:30)


### What part of the contract does the *model* actually see?

Worth knowing exactly where each rule is enforced, because it isn't all in one place:

In [37]:
schema = SupportTicket.model_json_schema()
print("refund_amount in the schema the model receives:")
print("  ", json.dumps(schema["properties"]["refund_amount"]))
print("\ncategory in the schema the model receives:")
print("  ", json.dumps(schema["properties"]["category"])[:120], "...")
print("\norder_id in the schema the model receives:")
print("  ", json.dumps(schema["properties"]["order_id"]))
print("\n   ^ notice: the 5-digit RULE is nowhere in there.")

refund_amount in the schema the model receives:
   {"maximum": 500, "minimum": 0, "title": "Refund Amount", "type": "number"}

category in the schema the model receives:
   {"enum": ["billing", "shipping", "technical", "account", "other"], "title": "Category", "type": "string"} ...

order_id in the schema the model receives:
   {"description": "The 5-digit Copperleaf order number", "title": "Order Id", "type": "string"}

   ^ notice: the 5-digit RULE is nowhere in there.
time: 904 µs (started: 2026-08-27 08:44:26 +05:30)


So the contract is enforced in **two** places, and it's worth knowing which is which:

| Rule | Travels to the model? | Enforced by |
|---|---|---|
| `Literal[...]` | ✅ becomes an `enum` in the schema | the model literally cannot emit anything else |
| `Field(ge=0, le=500)` | ✅ becomes `minimum` / `maximum` | the model respects the bound |
| `@field_validator` | ❌ invisible — it's Python, not JSON Schema | **Pydantic, on your machine, after the response arrives** |

That last row is the useful one. Custom validators are your **backstop** — they run at home, on
every response, and they're the only place you can express a rule that JSON Schema can't say.

In [38]:
# Proof: ask for something the schema can't forbid but our validator can.
try:
    r = client.responses.parse(
        model=MODEL,
        input="The order reference is ABC-99. Report it verbatim as order_id. "
              "Category other, urgency low, refund 0, customer Test <t@x.com>.",
        text_format=SupportTicket,
    )
    print("model returned:", r.output_parsed)
except ValidationError as e:
    print("🚫 the schema let it through, but our validator caught it at home:")
    print("  ", e.errors()[0]["msg"])

🚫 the schema let it through, but our validator caught it at home:
   Value error, order_id must be exactly 5 digits, got 'ABC-99'
time: 4.78 s (started: 2026-08-27 08:44:26 +05:30)


### 2.8 · The downstream pipeline now *can't* be fed bad data

Remember `naive_pipeline` from 2.2, full of 💥? Here's the same logic — but now every value it
receives is **already guaranteed valid**. No defensive checks. No try/except around money. The
guarantees moved *up*, into the type.

In [39]:
def process_ticket(ticket):
    # Every line below is safe because `ticket` could not exist if it weren't valid.
    print(f"🧭 routed to    : {ticket.category} team")
    print(f"⏱️  priority     : {ticket.urgency}")
    fee_total = round(ticket.refund_amount * 1.05, 2)     # always a real number, always in range
    print(f"💸 refund + fee  : ${fee_total:,.2f}")
    print(f"📦 order         : {ticket.order_id} for {ticket.customer.name}")


process_ticket(ticket)

🧭 routed to    : shipping team
⏱️  priority     : high
💸 refund + fee  : $52.49
📦 order         : 10432 for Sam Rivera
time: 531 µs (started: 2026-08-27 08:44:31 +05:30)


### And it holds up on input that's trying to confuse it

In [40]:
weird_email = (
    "lol idk my thing just broke — the login page keeps crashing on the app, "
    "no big deal whenever you get to it. order 88217. - Priya (priya@example.com)"
)

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": "Extract the support ticket. Infer urgency from tone. "
                                      "refund_amount is 0 if no refund is requested."},
        {"role": "user", "content": weird_email},
    ],
    text_format=SupportTicket,
)
print(response.output_parsed, "\n")
process_ticket(response.output_parsed)     # technical / low / $0.00 — routed right, no money moved

category='technical' urgency='low' refund_amount=0.0 order_id='88217' customer=Customer(name='Priya', email='priya@example.com') 

🧭 routed to    : technical team
⏱️  priority     : low
💸 refund + fee  : $0.00
📦 order         : 88217 for Priya
time: 6.73 s (started: 2026-08-27 08:44:31 +05:30)


## 2.9 · What structured outputs still do **not** give you

We now get a perfectly typed object every single time. Problem solved?

No. Look hard at where those *values* came from: **the model read them off the email.**
`refund_amount` is whatever the customer *claimed* they paid. `urgency` is a vibe. Nobody checked
the order exists, what it actually cost, or whether it was even delivered.

Here's a customer with a more creative memory:

In [41]:
liar_email = (
    "my order 50918 never turned up. I paid 480 dollars for that skillet and I want every "
    "cent back today. - Dale K (dale.k@example.com)"
)

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": "Extract the support ticket from the customer email."},
        {"role": "user", "content": liar_email},
    ],
    text_format=SupportTicket,
)
print(response.output_parsed, "\n")
process_ticket(response.output_parsed)

category='shipping' urgency='high' refund_amount=480.0 order_id='50918' customer=Customer(name='Dale K', email='dale.k@example.com') 

🧭 routed to    : shipping team
⏱️  priority     : high
💸 refund + fee  : $504.00
📦 order         : 50918 for Dale K
time: 11.7 s (started: 2026-08-27 08:44:37 +05:30)


That output is **flawless**. Valid category. Valid urgency. A float comfortably under our $500
ceiling. It sails through every rule we spent 2.4 and 2.5 writing, and `process_ticket` wires the
money without a murmur.

That skillet costs **$89.00**, and it's still **in transit**.

> ### A valid shape is not a true value.
>
> Pydantic guarantees the *form* of the answer. It has absolutely nothing to say about whether the
> answer is **true**, because the only source of facts in that call was the customer's own email.

To check the price, we'd have to look it up. To check delivery, we'd have to ask the carrier. To
check the refund window, we'd have to consult the policy.

Which is to say: **we need tools.** And now we have both halves.

---

P0 setup  ·  P1 function calling  ·  P2 structured outputs  ·  **P3 ▶ CASE STUDY**  ·  P4 wrap-up

> *both at once  —  one call, tools in, typed decision out*

# Part 3 · The Case Study — Copperleaf's support desk

We finished Part 1 with an agent that could **reach real facts** but only spoke prose.
We finished Part 2 with an agent that spoke **perfect typed data** but had no facts to speak about —
it just believed the customer.

Neither half is a system. Put them together and you have one:

```
        PART 1                      THE DESK                     PART 2
   ┌──────────────┐          ┌──────────────────┐        ┌──────────────────┐
   │  tools=[...] │  ──────▶ │                  │ ─────▶ │ text_format=     │
   │              │  facts   │      MODEL       │ decide │   TicketDecision │
   │ lookup_order │          │                  │        │                  │
   │ get_delivery │ ◀──────  │  round 1: asks   │        │  order_id  str   │
   │ check_policy │  asks    │  round 2: asks   │        │  action    enum  │
   │ get_history  │          │  round 3: decides│        │  refund    float │
   └──────────────┘          └──────────────────┘        └──────────────────┘
      your data                                             your contract

   ONE call to client.responses.parse(tools=..., text_format=...) does all of it.
```

The rule that makes this a *system* rather than a demo:

> **Every number in the final decision must come from a tool, not from the customer's email.**

## 3.1 · The back office

In production these would be database queries and carrier APIs. Here they're five dicts, so the
tool loop stays the thing you're looking at.

`TODAY` is pinned so the day-counts in this notebook don't drift as the months pass.

In [42]:
from datetime import date

TODAY = date(2026, 8, 27)          # pinned, so the numbers below never drift

ORDERS = {
    "10432": {"item": "Ceramic mug — Stoneware",  "price":  49.99, "ordered": "2026-08-14",
              "status": "delivered",  "delivered_on": "2026-08-18"},
    "88217": {"item": "Smart kettle base",        "price": 129.00, "ordered": "2026-07-30",
              "status": "delivered",  "delivered_on": "2026-08-02"},
    "77310": {"item": "Linen napkin set (6)",     "price":  34.50, "ordered": "2026-06-11",
              "status": "delivered",  "delivered_on": "2026-06-15"},
    "50918": {"item": "Cast-iron skillet, 12in",  "price":  89.00, "ordered": "2026-08-23",
              "status": "in_transit", "delivered_on": None},
    "62145": {"item": "Copper saucepan, 2L",      "price":  76.00, "ordered": "2026-08-20",
              "status": "delivered",  "delivered_on": "2026-08-24"},
}

CUSTOMERS = {
    "sam.rivera@example.com": {"name": "Sam Rivera",  "orders":  7, "prior_refunds": 1, "prior_refund_total":  22.00},
    "priya@example.com":      {"name": "Priya Nair",  "orders":  3, "prior_refunds": 0, "prior_refund_total":   0.00},
    "m.bell@example.com":     {"name": "Marcus Bell", "orders": 12, "prior_refunds": 0, "prior_refund_total":   0.00},
    "dale.k@example.com":     {"name": "Dale K",      "orders":  4, "prior_refunds": 3, "prior_refund_total": 410.00},
    "jun.park@example.com":   {"name": "Jun Park",    "orders":  2, "prior_refunds": 0, "prior_refund_total":   0.00},
}

# Copperleaf's published returns policy: how many days after delivery you may still claim.
REFUND_WINDOW_DAYS = {"damaged": 30, "faulty": 90, "not_delivered": 60, "change_of_mind": 14}

print(len(ORDERS), "orders,", len(CUSTOMERS), "customers, policy:", REFUND_WINDOW_DAYS)

5 orders, 5 customers, policy: {'damaged': 30, 'faulty': 90, 'not_delivered': 60, 'change_of_mind': 14}
time: 722 µs (started: 2026-08-27 08:44:49 +05:30)


### The four tools

Read `lookup_order` carefully and skim the rest — they're the same shape as everything in Part 1.

Note what `check_refund_policy` needs: a **`days_since_delivery`** number. The model can't invent
that; it has to get it from `get_delivery_status` first. That dependency is what forces a real
multi-round loop rather than one parallel burst.

In [43]:
def lookup_order(order_id):
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": False, "note": f"No Copperleaf order {order_id}."}
    return {"found": True, "order_id": order_id, "item": order["item"],
            "price": order["price"], "ordered": order["ordered"], "status": order["status"]}


def get_delivery_status(order_id):
    order = ORDERS.get(order_id)
    if order is None:
        return {"found": False, "note": f"No Copperleaf order {order_id}."}
    if order["delivered_on"] is None:
        return {"found": True, "status": order["status"], "delivered_on": None,
                "days_since_delivery": None, "note": "Not delivered yet — still with the carrier."}
    delivered = date.fromisoformat(order["delivered_on"])
    return {"found": True, "status": order["status"], "delivered_on": order["delivered_on"],
            "days_since_delivery": (TODAY - delivered).days}


def check_refund_policy(reason, days_since_delivery):
    window = REFUND_WINDOW_DAYS[reason]
    if days_since_delivery is None:
        return {"reason": reason, "window_days": window, "days_since_delivery": None,
                "eligible": False, "note": "Not delivered yet — the refund window has not started."}
    eligible = days_since_delivery <= window
    return {"reason": reason, "window_days": window,
            "days_since_delivery": days_since_delivery, "eligible": eligible,
            "note": ("Within the window." if eligible
                     else f"Claim is {days_since_delivery - window} days past the {window}-day window.")}


def get_customer_history(email):
    person = CUSTOMERS.get(email)
    if person is None:
        return {"found": False, "note": "No account for that address."}
    return {"found": True, **person,
            "serial_refunder": person["prior_refunds"] >= 3}


# Quick smoke test — no model involved.
print(lookup_order("50918"))
print(get_delivery_status("77310"))
print(check_refund_policy("change_of_mind", 73))
print(get_customer_history("dale.k@example.com"))

{'found': True, 'order_id': '50918', 'item': 'Cast-iron skillet, 12in', 'price': 89.0, 'ordered': '2026-08-23', 'status': 'in_transit'}
{'found': True, 'status': 'delivered', 'delivered_on': '2026-06-15', 'days_since_delivery': 73}
{'reason': 'change_of_mind', 'window_days': 14, 'days_since_delivery': 73, 'eligible': False, 'note': 'Claim is 59 days past the 14-day window.'}
{'found': True, 'name': 'Dale K', 'orders': 4, 'prior_refunds': 3, 'prior_refund_total': 410.0, 'serial_refunder': True}
time: 1.16 ms (started: 2026-08-27 08:44:49 +05:30)


In [44]:
DESK_TOOLS = [
    {
        "type": "function",
        "name": "lookup_order",
        "description": "Look up a Copperleaf order. Returns the real item, the real price paid, "
                       "the order date and the current status. Use this before quoting any amount.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "The 5-digit order number"}},
            "required": ["order_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_delivery_status",
        "description": "Get the carrier status for an order: whether it was delivered, on what date, "
                       "and how many days ago. Returns days_since_delivery = null if not yet delivered.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "The 5-digit order number"}},
            "required": ["order_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "check_refund_policy",
        "description": "Check Copperleaf's returns policy. Tells you whether a claim is still inside "
                       "the refund window. Requires the REAL days_since_delivery from get_delivery_status.",
        "parameters": {
            "type": "object",
            "properties": {
                "reason": {"type": "string", "description": "Why the customer wants a refund",
                           "enum": ["damaged", "faulty", "not_delivered", "change_of_mind"]},
                "days_since_delivery": {"type": ["number", "null"],
                                        "description": "Days since delivery, from get_delivery_status. "
                                                       "Pass null if the order is not delivered yet."},
            },
            "required": ["reason", "days_since_delivery"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_customer_history",
        "description": "Look up the sender's account: how many orders, how many previous refunds, "
                       "and whether they are flagged as a serial refunder.",
        "parameters": {
            "type": "object",
            "properties": {"email": {"type": "string", "description": "The customer's email address"}},
            "required": ["email"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

DESK_DISPATCH = {
    "lookup_order": lookup_order,
    "get_delivery_status": get_delivery_status,
    "check_refund_policy": check_refund_policy,
    "get_customer_history": get_customer_history,
}

print("Desk tools:", list(DESK_DISPATCH))

Desk tools: ['lookup_order', 'get_delivery_status', 'check_refund_policy', 'get_customer_history']
time: 771 µs (started: 2026-08-27 08:44:49 +05:30)


## 3.2 · Stage one — give the desk facts, and nothing else

`run_tool_loop` from 1.7, completely unchanged, with the desk's tools instead of the calculator's.
No `text_format` yet — so we're back to prose out.

We'll use Dale's email, the one that fooled Part 2 into wiring $480.

In [45]:
DESK_POLICY = (
    "You are Copperleaf's support desk. Decide what to do with the customer's email.\n"
    "\n"
    "CATEGORY — what the email is ABOUT:\n"
    "  billing   = charges, payments, double charges, the wrong card\n"
    "  shipping  = delivery, damage in transit, returns and refunds of physical goods\n"
    "  technical = the app or the website is broken\n"
    "  account   = password, profile or subscription problems\n"
    "  other     = anything else\n"
    "\n"
    "RULES, in order:\n"
    "1. Only choose action='refund' if the customer is actually asking for money back, a return or "
    "a replacement. A bug report, a question or a complaint with no refund request is "
    "action='reply_only' with refund_amount 0.\n"
    "2. NEVER trust a price, a total or a delivery date stated by the customer. Call lookup_order "
    "for the real price and get_delivery_status for the real delivery date.\n"
    "3. Before refunding anything, call check_refund_policy with the reason and the REAL "
    "days_since_delivery. If it says not eligible, refund_amount MUST be 0 AND action MUST be "
    "'reply_only' — never 'refund' with a zero amount.\n"
    "4. refund_amount may never exceed the real price from lookup_order.\n"
    "5. If get_delivery_status says the order is not delivered yet, do not refund: the claim is "
    "premature. Set refund_amount to 0 and do not use action='refund'.\n"
    "6. Always call get_customer_history for the sender. If serial_refunder is true, the action is "
    "escalate_to_human, whatever else you concluded.\n"
    "7. Billing disputes (double charges, the wrong card) are always escalate_to_human with "
    "refund_amount 0.\n"
    "8. Put every fact you relied on into `evidence`, naming the tool it came from."
)

print("Dale's email:\n ", liar_email, "\n")
print("Tool calls the desk made:")
response = run_tool_loop(
    liar_email,
    tools=DESK_TOOLS,
    dispatch=DESK_DISPATCH,
    instructions=DESK_POLICY,
)
print()
pretty_print(response.output_text)

Dale's email:
  my order 50918 never turned up. I paid 480 dollars for that skillet and I want every cent back today. - Dale K (dale.k@example.com) 

Tool calls the desk made:


   round 1: lookup_order({'order_id': '50918'}) -> {'found': True, 'order_id': '50918', 'item': 'Cast-iron skillet, 12in', 'price': 89.0, 'ordered': '2026-08-23', 'status': 'in_transit'}
   round 1: get_delivery_status({'order_id': '50918'}) -> {'found': True, 'status': 'in_transit', 'delivered_on': None, 'days_since_delivery': None, 'note': 'Not delivered yet — still with the carrier.'}
   round 1: get_customer_history({'email': 'dale.k@example.com'}) -> {'found': True, 'name': 'Dale K', 'orders': 4, 'prior_refunds': 3, 'prior_refund_total': 410.0, 'serial_refunder': True}


   round 2: check_refund_policy({'reason': 'not_delivered', 'days_since_delivery': None}) -> {'reason': 'not_delivered', 'window_days': 60, 'days_since_delivery': None, 'eligible': False, 'note': 'Not delivered yet — the refund window has not started.'}



Hi Dale,  Thanks for reaching out. I’ve started a check on order 50918 and
here’s what I found:  What I’m seeing - Order: 50918, item Cast-iron skillet,
12in - Real price recorded (by our system): $89.00 - Ordered: 2026-08-23 -
Current status: in transit (not delivered yet) - Carrier status: not delivered
yet - Your account history: serial_refunder is true (you’ve had multiple prior
refunds)  Refund eligibility and next steps - Our policy requires a human review
if the customer has a serial-refunder history. Because you’re flagged as a
serial refunder, I’m escalating this to a human agent for review. I cannot
approve an automatic refund in this case. - The refund policy check for a
“not_delivered” claim indicates the refund window hasn’t started yet since the
package hasn’t arrived, so an automatic refund wouldn’t be approved without a
human review. - The amount you asked for ($480) does not match the real order
price in our records ($89). We will base any potential refund on the actu

Watch the round numbers in that trace. `check_refund_policy` could only be called **after**
`get_delivery_status` came back — the model chained them, exactly like the calculator did in 1.6.

And the conclusion is now **right**: no $480. But it's right *in a paragraph*. Copperleaf's refund
processor still can't read it.

## 3.3 · Stage two — the decision contract

`SupportTicket` from Part 2, grown up. Same idea, four additions that a real desk needs:

- **`action`** — a `Literal`, because the router has exactly four things it knows how to do
- **`evidence`** — a list of the facts the tools returned. This is the field that makes the whole
  thing auditable: it's where "how did you reach $0?" gets answered.
- **`reply_to_customer`** — prose *is* allowed, it just has to live in a **named field** instead of
  being the entire response
- the 5-digit **validator** and the nested **`Customer`**, carried over unchanged from 2.5

In [46]:
class TicketDecision(BaseModel):
    order_id: str = Field(description="The 5-digit Copperleaf order number")
    category: Literal["billing", "shipping", "technical", "account", "other"]
    urgency: Literal["low", "medium", "high"]
    action: Literal["refund", "replace", "escalate_to_human", "reply_only"]
    refund_amount: float = Field(ge=0, le=500, description="0 unless policy allows a refund")
    evidence: list[str] = Field(description="Facts from tool results that justify this decision, "
                                            "each naming the tool it came from")
    reply_to_customer: str = Field(description="Two or three sentences, warm and specific")
    customer: Customer

    @field_validator("order_id")
    @classmethod
    def must_be_5_digits(cls, v):
        if not (v.isdigit() and len(v) == 5):
            raise ValueError(f"order_id must be exactly 5 digits, got {v!r}")
        return v


print(json.dumps(TicketDecision.model_json_schema()["properties"]["action"]))

{"enum": ["refund", "replace", "escalate_to_human", "reply_only"], "title": "Action", "type": "string"}
time: 1.78 ms (started: 2026-08-27 08:45:21 +05:30)


## 3.4 · Both at once — the hinge

Now the cell this whole notebook has been walking towards. **One** call, carrying **both** halves:

```python
client.responses.parse(
    tools=DESK_TOOLS,             # Part 1 — how facts get in
    text_format=TicketDecision,   # Part 2 — how decisions get out
)
```

Let's do the first round by hand, because what comes back is the most instructive thing here.

In [47]:
response = client.responses.parse(
    model=MODEL,
    instructions=DESK_POLICY,
    input=[{"role": "user", "content": liar_email}],
    tools=DESK_TOOLS,
    text_format=TicketDecision,
)

print("output items :", [item.type for item in response.output])
print("output_parsed:", response.output_parsed)
print()
for item in response.output:
    if item.type == "function_call":
        print(f"   it wants: {item.name}({item.arguments})")

output items : ['reasoning', 'function_call', 'function_call', 'function_call']
output_parsed: None

   it wants: lookup_order({"order_id":"50918"})
   it wants: get_delivery_status({"order_id":"50918"})
   it wants: get_customer_history({"email":"dale.k@example.com"})
time: 5.03 s (started: 2026-08-27 08:45:21 +05:30)


🎯 **`output_parsed` is `None`.**

That single line is the join between the two halves of this notebook. The model was handed a schema
*and* a set of tools, and it decided it is **not ready to fill the schema in yet** — it needs facts
first. So instead of a `TicketDecision`, we got a `function_call`.

`output_parsed` stays `None` for exactly as long as the model is still gathering. The moment it has
what it needs, the same field holds a fully validated object.

Let's feed it what it asked for and keep going.

In [48]:
tool_outputs = []
for call in response.output:
    if call.type != "function_call":
        continue
    args = json.loads(call.arguments)
    result = DESK_DISPATCH[call.name](**args)
    print(f"   we ran {call.name}({args})\n      -> {result}")
    tool_outputs.append(
        {"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)}
    )

response = client.responses.parse(
    model=MODEL,
    instructions=DESK_POLICY,
    previous_response_id=response.id,
    input=tool_outputs,
    tools=DESK_TOOLS,
    text_format=TicketDecision,
)

print("\noutput items :", [item.type for item in response.output])
print("output_parsed:", type(response.output_parsed).__name__ if response.output_parsed else None)

   we ran lookup_order({'order_id': '50918'})
      -> {'found': True, 'order_id': '50918', 'item': 'Cast-iron skillet, 12in', 'price': 89.0, 'ordered': '2026-08-23', 'status': 'in_transit'}
   we ran get_delivery_status({'order_id': '50918'})
      -> {'found': True, 'status': 'in_transit', 'delivered_on': None, 'days_since_delivery': None, 'note': 'Not delivered yet — still with the carrier.'}
   we ran get_customer_history({'email': 'dale.k@example.com'})
      -> {'found': True, 'name': 'Dale K', 'orders': 4, 'prior_refunds': 3, 'prior_refund_total': 410.0, 'serial_refunder': True}



output items : ['reasoning', 'function_call']
output_parsed: None
time: 5.85 s (started: 2026-08-27 08:45:26 +05:30)


Still gathering, or already decided — depends how many facts it wanted. Either way you can see the
mechanism: **keep looping until `output_parsed` stops being `None`.**

Which is precisely what `run_tool_loop` already does. We wrote it in 1.7 for a calculator; it needs
**zero changes** to run a support desk.

## 3.5 · The desk, in one call

In [49]:
print("Dale's email:\n ", liar_email, "\n")
print("Tool calls:")
response = run_tool_loop(
    liar_email,
    tools=DESK_TOOLS,
    dispatch=DESK_DISPATCH,
    instructions=DESK_POLICY,
    text_format=TicketDecision,          # <- the only difference from 3.2
)

decision = response.output_parsed
print("\n──────── DECISION ────────")
print("order      :", decision.order_id)
print("category   :", decision.category)
print("urgency    :", decision.urgency)
print("action     :", decision.action)
print("refund     : $", decision.refund_amount)
print("customer   :", decision.customer.name, "<" + decision.customer.email + ">")
print("\nevidence:")
for fact in decision.evidence:
    print("   •", fact)
print("\nreply:")
pretty_print(decision.reply_to_customer)

Dale's email:
  my order 50918 never turned up. I paid 480 dollars for that skillet and I want every cent back today. - Dale K (dale.k@example.com) 

Tool calls:


   round 1: lookup_order({'order_id': '50918'}) -> {'found': True, 'order_id': '50918', 'item': 'Cast-iron skillet, 12in', 'price': 89.0, 'ordered': '2026-08-23', 'status': 'in_transit'}
   round 1: get_delivery_status({'order_id': '50918'}) -> {'found': True, 'status': 'in_transit', 'delivered_on': None, 'days_since_delivery': None, 'note': 'Not delivered yet — still with the carrier.'}
   round 1: get_customer_history({'email': 'dale.k@example.com'}) -> {'found': True, 'name': 'Dale K', 'orders': 4, 'prior_refunds': 3, 'prior_refund_total': 410.0, 'serial_refunder': True}



──────── DECISION ────────
order      : 50918
category   : shipping
urgency    : high
action     : escalate_to_human
refund     : $ 0.0
customer   : Dale K <dale.k@example.com>

evidence:
   • lookup_order tool: order 50918 found; item Cast-iron skillet, 12in; price_paid (from system) 89.0; status in_transit; ordered 2026-08-23
   • get_delivery_status tool: not delivered yet — still with the carrier; days_since_delivery: null
   • get_customer_history tool: serial_refunder true; orders: 4; prior_refunds: 3; prior_refund_total: 410.0; name: Dale K

reply:
Hi Dale, I’m sorry your skillet hasn’t arrived yet. I’ve escalated this case to
a human specialist for immediate review. We’ll follow up with next steps and any
refund decisions once the shipment status is confirmed.
time: 25.6 s (started: 2026-08-27 08:45:32 +05:30)


Compare that to 2.9, where the identical email produced a flawless, fully-typed instruction to wire
**$480**.

Same model. Same schema. Same customer. The difference is that this time the model had to **go and
look**, and the `evidence` list shows you exactly what it found. That list is the thing you show an
auditor — or a customer who disagrees.

## 3.6 · The whole inbox

Five emails, one desk. This is what the system actually looks like in service.

In [50]:
INBOX = [
    customer_email,        # Sam Rivera  — 10432, smashed mug, damaged, 9 days ago
    weird_email,           # Priya Nair  — 88217, app crashing, no refund wanted
    ("I changed my mind about the linen napkins from order 77310 — they don't match the table "
     "at all. Can I send them back for a refund please? - Marcus Bell (m.bell@example.com)"),
    liar_email,            # Dale K      — 50918, claims $480 for an $89 skillet in transit
    ("I've been charged twice for order 62145. There are two separate 76 dollar charges on my "
     "card from Copperleaf on the same day. Please fix this. - Jun Park (jun.park@example.com)"),
]

decisions = []
for i, email in enumerate(INBOX, 1):
    print(f"\n═════ ticket {i} " + "═" * 50)
    print(textwrap.fill(email, width=88))
    print("  tools:")
    response = run_tool_loop(
        email,
        tools=DESK_TOOLS,
        dispatch=DESK_DISPATCH,
        instructions=DESK_POLICY,
        text_format=TicketDecision,
    )
    decisions.append(response.output_parsed)


═════ ticket 1 ══════════════════════════════════════════════════
hey, my order 10432 turned up completely smashed — the ceramic mug is in pieces. I paid
49.99 for it and honestly I'm pretty annoyed, this is the second time. I just want my
money back asap, can you sort this out today?  Thanks, Sam Rivera
(sam.rivera@example.com)
  tools:


   round 1: lookup_order({'order_id': '10432'}) -> {'found': True, 'order_id': '10432', 'item': 'Ceramic mug — Stoneware', 'price': 49.99, 'ordered': '2026-08-14', 'status': 'delivered'}
   round 1: get_delivery_status({'order_id': '10432'}) -> {'found': True, 'status': 'delivered', 'delivered_on': '2026-08-18', 'days_since_delivery': 9}
   round 1: get_customer_history({'email': 'sam.rivera@example.com'}) -> {'found': True, 'name': 'Sam Rivera', 'orders': 7, 'prior_refunds': 1, 'prior_refund_total': 22.0, 'serial_refunder': False}


   round 2: check_refund_policy({'reason': 'damaged', 'days_since_delivery': 9}) -> {'reason': 'damaged', 'window_days': 30, 'days_since_delivery': 9, 'eligible': True, 'note': 'Within the window.'}



═════ ticket 2 ══════════════════════════════════════════════════
lol idk my thing just broke — the login page keeps crashing on the app, no big deal
whenever you get to it. order 88217. - Priya (priya@example.com)
  tools:


   round 1: lookup_order({'order_id': '88217'}) -> {'found': True, 'order_id': '88217', 'item': 'Smart kettle base', 'price': 129.0, 'ordered': '2026-07-30', 'status': 'delivered'}
   round 1: get_delivery_status({'order_id': '88217'}) -> {'found': True, 'status': 'delivered', 'delivered_on': '2026-08-02', 'days_since_delivery': 25}
   round 1: get_customer_history({'email': 'priya@example.com'}) -> {'found': True, 'name': 'Priya Nair', 'orders': 3, 'prior_refunds': 0, 'prior_refund_total': 0.0, 'serial_refunder': False}


...


═════ ticket 4 ══════════════════════════════════════════════════
my order 50918 never turned up. I paid 480 dollars for that skillet and I want every
cent back today. - Dale K (dale.k@example.com)
  tools:


   round 1: lookup_order({'order_id': '50918'}) -> {'found': True, 'order_id': '50918', 'item': 'Cast-iron skillet, 12in', 'price': 89.0, 'ordered': '2026-08-23', 'status': 'in_transit'}
   round 1: get_delivery_status({'order_id': '50918'}) -> {'found': True, 'status': 'in_transit', 'delivered_on': None, 'days_since_delivery': None, 'note': 'Not delivered yet — still with the carrier.'}
   round 1: get_customer_history({'email': 'dale.k@example.com'}) -> {'found': True, 'name': 'Dale K', 'orders': 4, 'prior_refunds': 3, 'prior_refund_total': 410.0, 'serial_refunder': True}



═════ ticket 5 ══════════════════════════════════════════════════
I've been charged twice for order 62145. There are two separate 76 dollar charges on my
card from Copperleaf on the same day. Please fix this. - Jun Park (jun.park@example.com)
  tools:


   round 1: lookup_order({'order_id': '62145'}) -> {'found': True, 'order_id': '62145', 'item': 'Copper saucepan, 2L', 'price': 76.0, 'ordered': '2026-08-20', 'status': 'delivered'}
   round 1: get_delivery_status({'order_id': '62145'}) -> {'found': True, 'status': 'delivered', 'delivered_on': '2026-08-24', 'days_since_delivery': 3}
   round 1: get_customer_history({'email': 'jun.park@example.com'}) -> {'found': True, 'name': 'Jun Park', 'orders': 2, 'prior_refunds': 0, 'prior_refund_total': 0.0, 'serial_refunder': False}


time: 2min 1s (started: 2026-08-27 08:45:58 +05:30)


In [51]:
print(f"{'order':<8}{'category':<11}{'urg':<7}{'action':<20}{'refund':>9}   customer")
print("─" * 76)
for d in decisions:
    print(f"{d.order_id:<8}{d.category:<11}{d.urgency:<7}{d.action:<20}"
          f"${d.refund_amount:>8,.2f}   {d.customer.name}")

print(f"\nTotal Copperleaf is about to pay out: ${sum(d.refund_amount for d in decisions):,.2f}")

order   category   urg    action                 refund   customer
────────────────────────────────────────────────────────────────────────────
10432   shipping   high   refund              $   49.99   Sam Rivera
88217   technical  low    reply_only          $    0.00   Priya Nair
77310   shipping   low    reply_only          $    0.00   Marcus Bell
50918   shipping   high   escalate_to_human   $    0.00   Dale K
62145   billing    high   escalate_to_human   $    0.00   Jun Park

Total Copperleaf is about to pay out: $49.99
time: 460 µs (started: 2026-08-27 08:47:59 +05:30)


Every row in that table is a **typed object**, not a sentence. `process_ticket` from 2.8 could
consume any of them without a single defensive check — and every figure in the `refund` column came
out of `ORDERS`, not out of a customer's memory.

## 3.7 · Where the money was actually saved

Two of those five tickets are the interesting ones. Let's look at what the tools did to them.

In [52]:
for d in decisions:
    if d.order_id in ("77310", "50918"):
        real_price = ORDERS[d.order_id]["price"]
        print(f"\n── order {d.order_id} — {ORDERS[d.order_id]['item']} (really ${real_price}) ──")
        print(f"   action : {d.action}    refund: ${d.refund_amount:,.2f}")
        for fact in d.evidence:
            print("   •", fact)


── order 77310 — Linen napkin set (6) (really $34.5) ──
   action : reply_only    refund: $0.00
   • lookup_order: order_id=77310, item='Linen napkin set (6)', price=34.5, ordered=2026-06-11, status='delivered'
   • get_delivery_status: order_id=77310, delivered_on=2026-06-15, days_since_delivery=73
   • check_refund_policy: reason='change_of_mind', days_since_delivery=73, eligible=false, window_days=14
   • get_customer_history: email='m.bell@example.com', name='Marcus Bell', orders=12, prior_refunds=0, serial_refunder=false

── order 50918 — Cast-iron skillet, 12in (really $89.0) ──
   action : escalate_to_human    refund: $0.00
   • lookup_order: found order 50918; item Cast-iron skillet 12in; price 89.0; ordered 2026-08-23; status in_transit
   • get_delivery_status: not delivered yet; days_since_delivery: null; note: 'Not delivered yet – still with the carrier'
   • get_customer_history: serial_refunder true; prior_refunds 3; prior_refund_total 410.0
time: 304 µs (started: 2026-0

- **77310** — Marcus genuinely bought the napkins, and genuinely wants to return them. But
  `get_delivery_status` said 73 days, and `check_refund_policy` said the change-of-mind window is
  14. Nothing about that email is dishonest; the *policy* is what says no. A Part-2-only desk would
  have refunded it, because the email reads perfectly reasonable.
- **50918** — Dale claimed $480 for an $89 skillet that is **still with the carrier**, and
  `get_customer_history` flags him at 3 prior refunds. Three separate tools each caught something.

### The layers, and what each one actually stops

| Layer | Stops | Where it runs |
|---|---|---|
| `lookup_order` (tool) | the invented $480 | your code |
| `get_delivery_status` + `check_refund_policy` (tools) | out-of-window and premature claims | your code |
| `get_customer_history` (tool) | serial refunders | your code |
| `Literal[...]` (schema) | invented categories and actions | the model, enforced by the API |
| `Field(ge=0, le=500)` (schema) | the 100× decimal slip | the model, enforced by the API |
| `@field_validator` (Pydantic) | malformed order ids | **your machine, after the response** |

Tools police the **facts**. The schema polices the **shape**. You need both, and they fail
differently — which is exactly why having both is not redundant.

---

P0 setup  ·  P1 function calling  ·  P2 structured outputs  ·  P3 case study  ·  **P4 ▶ WRAP-UP**

> *what each half actually bought you*

# Part 4 · Wrap-up

### The thread we followed

1. The model **couldn't reach our world** — no weather, no arithmetic, no actions (§1.1).
2. We gave it **tools**, and it learned to ask for help — one call, then many, then chained across
   rounds (§1.4–1.7).
3. But its answers were still **prose**, unusable downstream (§1.10).
4. We demanded JSON and hand-parsed it — fragile, and one silent decimal slip wired **$5,249** (§2.2).
5. We wrote the shape down **once** as a Pydantic class and validated against it (§2.3–2.6).
6. We handed that same class to the API with `text_format=`, so the model is **forced** to fill it
   in (§2.7).
7. But a valid shape is **not a true value** — the schema happily approved a $480 refund on an $89
   skillet (§2.9).
8. So we put **both in one call**: tools to fetch the truth, schema to shape the decision (§3.4–3.6).

### Every failure mode, and what fixed it

| Failure | Without | With |
|---|---|---|
| *"the weather is probably 25°C"* | confident invention | `get_weather` → the real number |
| `1247 * 83` off by thousands | the model guessing | `multiply` → exact |
| prose / <code>```json</code> fences | `json.loads` crashes at random | typed object, always |
| `refund_amount = "49.99"` | math explodes | coerced to `float`, or rejected |
| `refund_amount = 4999` | **$5,249 wired** | `Field(le=500)` |
| `category = "damaged_item"` | silent misroute | `Literal` — unrepresentable |
| missing field | `KeyError` deep in the pipeline | rejected up front |
| `order_id = "ABC"` | bad lookups downstream | `@field_validator`, client-side |
| **customer claims $480 for an $89 item** | **paid in full, politely** | `lookup_order` |
| **refund claimed 73 days after a 14-day window** | **paid in full** | `check_refund_policy` |
| serial refunder | paid again | `get_customer_history` |

### The one sentence

> **Tools decide what the model *knows*. Schemas decide what your code *receives*. Ship neither and
> you have a chatbot; ship one and you have a liability; ship both and you have a system.**

### When to reach for which

| | Reach for it when |
|---|---|
| 🔧 **Function calling** | the answer depends on facts the model can't have — your DB, live data, another service — or the model needs to *do* something |
| 📦 **Structured outputs** | the output feeds another system: a router, a queue, a DB write, a payment. Also any extraction task (resumes, invoices, emails → fields) |
| 🔗 **Both** | the model must *look something up* before it can *decide* something — which is most real work |
| 🚫 **Neither** | free-form prose meant only for a human to read |

### Things to try from here

- Add a `replace_item` tool and a `stock_level` tool, and see whether the desk starts choosing
  `action="replace"` over `refund` when stock allows.
- Break rule 2 in `DESK_POLICY` (delete the "NEVER trust a price" line) and re-run Dale's email.
  Watch how quickly the $480 comes back.
- Delete rule 1 and re-run Priya's ticket. She never asked for a refund — without that rule the desk
  cheerfully sends her $129 for a login bug. Silent over-refunding is the failure mode nobody
  notices, because no exception is ever raised.
- Add `confidence: float = Field(ge=0, le=1)` to `TicketDecision` and route anything under 0.7 to a
  human.
- Point `lookup_order` at a real SQLite table. Nothing else in the notebook has to change — that's
  the same fake-to-real swap we did with `get_weather` in 1.8.